# Initial ML solution: supervised evidence selection

Three experiments on the **existing BM25 v2 top-20 train/dev rankings** (no 16-million-passage re-encoding): **1** a trainable linear projection over frozen BGE embeddings; **2** a task-fine-tuned MiniLM cross-encoder; **3** the cross-encoder followed by distinct-source selection and a matched DistilBERT verifier. The third is our candidate *main method*, not an established improvement.

Run cells in order on a Colab GPU. All artifacts and resumable checkpoints live in Drive. Experiment runners execute directly in this notebook process, so tqdm progress bars display inline. You may rerun a cell after interruption. Do **not** change settings within a run name; use a new output root if you change them. Push this notebook and its three new scripts to GitHub before Colab setup.

Training relevance is a **source-URL proxy**: a positive chunk shares a URL with an annotated evidence source. The label may not mean the particular chunk is decisive. Gold URLs and dev qrels are never prediction-time features. Official test is untouched. A top-20 reranker cannot recover evidence outside the saved top 20 (BM25 proxy hit@20 was 0.4223 on 386 judged dev claims).


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-supervised')
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has local edits; save them before pulling.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repo revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json, runpy, time
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, restart, and rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
TRAIN_CLAIMS = PROCESSED / 'train.jsonl'
DEV_CLAIMS = PROCESSED / 'dev.jsonl'
TRAIN_BM25 = DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
ROOT = DATA_ROOT / 'outputs/initial_ml_solution_supervised_rerank_v1'
PROJECTION = ROOT / 'projection/bge_linear_v1'
CROSS = ROOT / 'cross_encoder/minilm_weak_v1'
DIVERSE_TRAIN = ROOT / 'source_diverse/train_cross_oof_v1'
DIVERSE_DEV = ROOT / 'source_diverse/dev_cross_v1'
VERIFIER = ROOT / 'verifier/cross_diverse_distilbert_v1'
required = [TRAIN_CLAIMS, DEV_CLAIMS, TRAIN_BM25, DEV_BM25, DEV_QRELS]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing previous artifacts:\n' + '\n'.join(missing))
for script in ('supervised_rerank_common.py', 'train_projection_reranker.py', 'train_cross_encoder_weak.py', 'run_source_diversity.py', 'train_transformer_verifier.py'):
    if not (REPO_DIR / 'scripts' / script).is_file():
        raise FileNotFoundError(f'Missing {script}; push local changes and rerun setup/pull.')
ROOT.mkdir(parents=True, exist_ok=True)
print('Inputs ready; output root:', ROOT)

def run_script(label, filename, arguments):
    script = REPO_DIR / 'scripts' / filename
    old_argv = sys.argv[:] 
    sys.argv = [str(script), *map(str, arguments)]
    print(f'\n=== {label} ===\n{script}\nArguments: {arguments}', flush=True)
    started = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = old_argv
        print(f'{label}: {((time.monotonic()-started)/60):.1f} minutes this invocation', flush=True)


## Experiment 1 — single trainable projection over frozen BGE

Caches only BM25 top-20 claim/passage embeddings in resumable 32-claim chunks. A linear layer, initialized as identity, is trained with listwise weak-positive loss. Learning rate, epochs, and BM25 blend are chosen on a **train-only** held-out 10%; the selected configuration is refit on annotated train claims, then dev is ranked once. No full-corpus dense retrieval. The output is a dev retrieval result, **not** a new end-to-end verdict model.


In [ ]:
run_script('Projection reranker', 'train_projection_reranker.py', [
    '--train-claims', TRAIN_CLAIMS, '--train-rankings', TRAIN_BM25,
    '--dev-rankings', DEV_BM25, '--dev-qrels', DEV_QRELS,
    '--output-dir', ROOT / 'projection', '--run-name', 'bge_linear_v1',
    '--top-n', 20, '--batch-size', 32, '--chunk-claims', 32,
    '--max-seq-length', 256, '--device', 'cuda',
])
assert (PROJECTION / 'dev_rankings.jsonl').is_file()
print('Projection retrieval:', json.loads((PROJECTION / 'dev_metrics.json').read_text())['dev'])


## Experiment 2 — task-fine-tuned cross-encoder

Fine-tunes the small MS MARCO MiniLM cross-encoder on one annotated-source candidate and two BM25-hard negatives per eligible train claim. The BM25/model score blend is selected by **train-only** proxy hit@3. Training saves model/optimizer state every 250 minibatches; an interrupted run resumes from that checkpoint. It also builds **two-fold out-of-fold train rankings** for Experiment 3, so no claim's retrieval label trains the model used to rank that claim. This is the most compute-heavy cell; allow for a long GPU run and rerun it after Colab interruption.


In [ ]:
run_script('Weakly supervised cross-encoder + OOF train rankings', 'train_cross_encoder_weak.py', [
    '--train-claims', TRAIN_CLAIMS, '--train-rankings', TRAIN_BM25,
    '--dev-rankings', DEV_BM25, '--dev-qrels', DEV_QRELS,
    '--output-dir', ROOT / 'cross_encoder', '--run-name', 'minilm_weak_v1',
    '--top-n', 20, '--negatives', 2, '--learning-rate', '2e-5',
    '--epochs', 2, '--batch-size', 16, '--inference-batch-size', 32,
    '--max-seq-length', 256, '--oof-folds', 2,
    '--checkpoint-steps', 250, '--checkpoint-claims', 20, '--device', 'cuda',
])
assert (CROSS / 'dev_rankings.jsonl').is_file() and (CROSS / 'train_oof_rankings.jsonl').is_file()
print('Cross-encoder retrieval:', json.loads((CROSS / 'dev_metrics.json').read_text())['dev'])


## Experiment 3 — main-method candidate: learned ranking + source diversity + matched verifier

Select the best-scoring chunk from each distinct source URL before filling the remaining slots. Do this to both **out-of-fold train** rankings and **frozen full-fit dev** rankings. Then train a fresh DistilBERT verdict model on the corresponding top-three evidence; its model hyperparameters are selected on an internal train split. This is an end-to-end result comparable with the previous BM25 + DistilBERT and Qwen baselines on the same 500 dev claims. We cannot claim it is better until the final cell reports the numbers.


In [ ]:
run_script('Source diversity / OOF train', 'run_source_diversity.py', [
    '--input-rankings', CROSS / 'train_oof_rankings.jsonl',
    '--output-dir', ROOT / 'source_diverse', '--run-name', 'train_cross_oof_v1',
    '--split', 'train', '--max-candidates', 20, '--metric-k', 3,
])
run_script('Source diversity / dev', 'run_source_diversity.py', [
    '--input-rankings', CROSS / 'dev_rankings.jsonl', '--qrels', DEV_QRELS,
    '--output-dir', ROOT / 'source_diverse', '--run-name', 'dev_cross_v1',
    '--split', 'dev', '--max-candidates', 20, '--metric-k', 3,
])
assert (DIVERSE_TRAIN / 'train_rankings.jsonl').is_file() and (DIVERSE_DEV / 'dev_rankings.jsonl').is_file()
print('Source-diverse retrieval:', json.loads((DIVERSE_DEV / 'dev_metrics.json').read_text())['diverse'])


In [ ]:
run_script('Matched DistilBERT verifier', 'train_transformer_verifier.py', [
    '--data-root', DATA_ROOT,
    '--train-rankings', DIVERSE_TRAIN / 'train_rankings.jsonl',
    '--eval-rankings', DIVERSE_DEV / 'dev_rankings.jsonl',
    '--output-dir', ROOT / 'verifier', '--run-name', 'cross_diverse_distilbert_v1',
    '--model', 'distilbert-base-uncased', '--max-length', 512,
    '--claim-token-budget', 96, '--evidence-token-budget', 110,
    '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
    '--batch-size', 8, '--gradient-accumulation-steps', 2,
    '--checkpoint-steps', 100, '--seed', 718, '--resume',
])
assert (VERIFIER / 'final_dev_metrics.json').is_file()
verdict = json.loads((VERIFIER / 'final_dev_metrics.json').read_text())
print('Main-method candidate:', {key: verdict[key] for key in ('macro_f1', 'accuracy', 'decision_error_rates')})


## Decision readout

Do not pick a model because of dev performance alone after trying several variants; treat this as exploratory and keep official test sealed. Compare retrieval on the same 386 URL-judged claims and verdicts on all 500 dev claims. A higher URL-proxy hit@3 must be checked against actual passage usefulness, macro-F1, and false reassurance. If the learned reranker loses to BM25, that is a valid finding. Before a deployment claim, manually audit a sample of promoted/demoted chunks.


In [ ]:
from scripts.run_bm25 import load_qrels
from src.retrieval.metrics import evaluate_rankings
from src.utils.io import read_jsonl
def retrieval_at_3(path):
    rows = read_jsonl(path)
    return evaluate_rankings({str(row['claim_id']): [str(item['passage_id']) for item in row['retrieved']] for row in rows}, load_qrels(DEV_QRELS), k=3)
for name, path in [
    ('BM25 v2', DEV_BM25), ('Projection', PROJECTION/'dev_rankings.jsonl'),
    ('Trained cross-encoder', CROSS/'dev_rankings.jsonl'),
    ('Cross-encoder + diversity', DIVERSE_DEV/'dev_rankings.jsonl'),
]:
    if path.is_file():
        m = retrieval_at_3(path)
        print(f"{name:28s} hit@3={m['evidence_hit_at_3']:.4f} MRR={m['mrr']:.4f} judged={m['judged_claim_count']}")
prior = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16/transformer/bm25_top3/final_dev_metrics.json'
claim_only = DATA_ROOT / 'outputs/initial_ml_solution_followup_v1/transformer/claim_only/final_dev_metrics.json'
for name, path in [('Prior BM25 + DistilBERT', prior), ('Claim-only DistilBERT diagnostic', claim_only), ('Cross-encoder + diversity + DistilBERT', VERIFIER/'final_dev_metrics.json')]:
    if path.is_file():
        m = json.loads(path.read_text())
        print(f"{name:43s} macro-F1={m['macro_f1']:.4f} accuracy={m['accuracy']:.4f} FR={m['decision_error_rates']['false_reassurance_rate']:.4f} FA={m['decision_error_rates']['false_alarm_rate']:.4f}")
